In [ ]:
# Jupyter Notebook : Nettoyage automatique des tables MySQL

import mysql.connector


In [ ]:
# Connexion MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="palworld_database"
)
cursor = conn.cursor(dictionary=True)

In [ ]:
# Fonction SELECT générique
def run_query(query):
    if cursor.nextset():  # si un résultat précédent n’est pas consommé
        cursor.fetchall()
    cursor.execute(query)
    return cursor.fetchall()

In [ ]:
# Lister les tables
print("\u2705 Tables dans la base :")
tables = run_query("SHOW TABLES;")
for t in tables:
    print("-", list(t.values())[0])

In [ ]:
# Analyse des valeurs manquantes
def check_nulls(table):
    print(f"\n\ud83d\udd0e Valeurs manquantes dans la table : {table}")
    cursor.execute(f"SELECT * FROM {table} LIMIT 1")
    cols = [desc[0] for desc in cursor.description]
    for col in cols:
        q = f"SELECT COUNT(*) AS null_count FROM {table} WHERE {col} IS NULL OR {col} = ''"
        result = run_query(q)
        if result[0]['null_count'] > 0:
            print(f"- {col} : {result[0]['null_count']} valeur(s) manquante(s)")

In [ ]:
# Exemple : analyse sur combat_attribute
check_nulls("combat_attribute")

In [ ]:
# Détection des doublons sur une colonne donnée
def check_duplicates(table, col):
    print(f"\n\ud83d\udcc9 Doublons dans {table} (colonne : {col})")
    q = f"""
    SELECT {col}, COUNT(*) AS cnt
    FROM {table}
    GROUP BY {col}
    HAVING cnt > 1
    """
    duplicates = run_query(q)
    for row in duplicates:
        print(f"- {row[col]} → {row['cnt']} doublons")

In [ ]:
# Exemple : doublons sur le nom
check_duplicates("combat_attribute", "name")


In [ ]:
# Normalisation (minuscule) d'une colonne texte
def normalize_column(table, col):
    q = f"UPDATE {table} SET {col} = LOWER({col}) WHERE {col} IS NOT NULL"
    cursor.execute(q)
    conn.commit()
    print(f"\n✅ Colonne {col} de {table} normalisée.")


In [ ]:
# Exemple : normaliser english_name
table = "job_skill"
col = "english_name"
normalize_column(table, col)


In [ ]:
# Fermer la connexion
cursor.close()
conn.close()